# Text representation

### What?
- converting texts into numbers
- extracting features from texts
- called as text vectorization
- core idea is to capture semantic meaning
- we need to do this because computers understand numbers only

### Why? :
- We extracts features from texts expressing them into number.
- Computers understand numbers only.
- if u give bad features to good model it will provide bad results
- it is seen that if u provide good features to mediate model it may perform better than above case of best model.

### 

### why is it difficult?
- image can be converted into numbers using pixels
- speech can be converted into numbers using amplitude of speech waveform w.r.t. time
- but how to convert text into numbers which carry semantic meaning.

### Core idea :- 
- is to express text into numbers which carry semantic meaning.

### Techniques :
- 1. OneHotEncoding
- 2. BOW
- 3. n-grams
- 4. Tf-Idf
- 5. custom features using domain knowledge
- 6. Word2Vec (embedding) in deep learning

### Common terms
- **Corpus**: combination of all the words in dataset (C)
- **Vocabulary**: Unique words in corpus(V)
- **Document**: Individual sentence (D)
- **Word**: every document consists of words, basic level of corpus.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [66]:
text = {"text": 
    ["what is NLP", "NLP is great", "NLP is popular", "NLP is NLP"], 
    "output":[1, 0, 1, 1]
    }

In [67]:
df = pd.DataFrame(text)

In [68]:
df.head()

,text,output
0,what is NLP,1
1,NLP is great,0
2,NLP is popular,1
3,NLP is NLP,1


**Here** 
- Corpus(c) : "what is  NLP NLP is great NLP is popular NLP is NLP"
- Vocabulary(v) : 5 unique words
- Documents(D) : 4 documents ex. D1 = "what is NLP" etc
- Words(W) : all the words like.. what, is, NLP, IS etc

## 1. OHE - OneHotEncoding

In [72]:
vocab = []
for sentence in df['text']:
    for word in sentence.lower().split():
        if word not in vocab:
            vocab.append(word)
vocab

['what', 'is', 'nlp', 'great', 'popular']

In [73]:
def OHE(sentence):
    document = []
    vector = np.zeros(len(vocab))
    for word in sentence.lower().split():
        if word not in vocab:
            document.append(vector)
        else:
            index = vocab.index(word)
            vector[index] = 1
            document.append(vector)
            vector = np.zeros(len(vocab))
    return document

In [74]:
len(vocab)

5

In [75]:
df['sentence_ohe'] = df['text'].apply(OHE)

In [76]:
df[["text", "sentence_ohe"]]

,text,sentence_ohe
0,what is NLP,"[[1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."
1,NLP is great,"[[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."
2,NLP is popular,"[[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."
3,NLP is NLP,"[[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."


In [77]:
arr = np.array(df["sentence_ohe"][0])

In [78]:
arr

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.]])

### Pros
- intuitive logic
- easy implementation

### Cons
- Sparsity , if i have 50k vocab then every word will be having 49999 zeros and 1 one. and it lead to overfitting in ML
- shape of every document is not fixed but ML algo assume fixed input
- in prediction if new word comes which is not there even in vocab , at that time it will ignore that word, but that word might have more weightage in that sentence.(OOB - Out Of Vocab)
- it doesn't capture semantic meaning at all

## 2. BOW - Bag of words

- based on frequency of words, how many times a word came in that document (remember sentence --> document)
- mostly used in text classification problems

In [79]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [80]:
bow = cv.fit_transform(df['text'])

In [81]:
cv.vocabulary_

{'what': 4, 'is': 1, 'nlp': 2, 'great': 0, 'popular': 3}

In [82]:
print(bow[0].toarray())
print(bow[1].toarray())
print(bow[2].toarray())
print(bow[3].toarray())

[[0 1 1 0 1]]
[[1 1 1 0 0]]
[[0 1 1 1 0]]
[[0 1 2 0 0]]


In [83]:
cv1 = CountVectorizer(binary=True)

In [84]:
cv1.fit_transform(df['text']).toarray()

array([[0, 1, 1, 0, 1],
       [1, 1, 1, 0, 0],
       [0, 1, 1, 1, 0],
       [0, 1, 1, 0, 0]])

- binary = True --> all non zero elements become 1 , better performing in sentiment analysis
- called binary BOW , it does count words it simpley if it present then 1 else 0

In [85]:
cv2 = CountVectorizer(max_features=1)

In [86]:
cv2.fit_transform(df['text']).toarray()

array([[1],
       [1],
       [1],
       [2]])

### Advantages
- intuitive
- solves fixed size problem of OHE
- OOV problem solved
- it captures semantic meaning at some extent 

### Disadvantages
- sparsity
- if Hello word comes in prediction which is not there in vocab so it says it zero but this word Hello may be the stronges word among all the words in a document so it is ignoring this word.
- we don't consider order of words
- according to BOW , You are good and You are bad. both are neighbour but in reality both are opposite.

# 3. N-grams

**Instead of forming vocabulary using one word only we consider N-words and form vocab.**

- Bi-grams - 2 words vocab
- tri-grams - 3 words vocab
- N-grams  - N words vocab

In [87]:
df.head()

,text,output,sentence_ohe
0,what is NLP,1,"[[1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."
1,NLP is great,0,"[[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."
2,NLP is popular,1,"[[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."
3,NLP is NLP,1,"[[0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0...."


In [88]:
cv_bigram = CountVectorizer(ngram_range = (2, 2))

In [90]:
cv_bigram.fit_transform(df['text']).toarray()

array([[0, 1, 0, 0, 1],
       [1, 0, 0, 1, 0],
       [0, 0, 1, 1, 0],
       [0, 1, 0, 1, 0]])

In [91]:
cv_bigram.vocabulary_

{'what is': 4, 'is nlp': 1, 'nlp is': 3, 'is great': 0, 'is popular': 2}

In [92]:
cv_unigram_bigram = CountVectorizer(ngram_range = (1, 2))

In [93]:
cv_unigram_bigram.fit_transform(df['text']).toarray()

array([[0, 1, 0, 1, 0, 1, 0, 0, 1, 1],
       [1, 1, 1, 0, 0, 1, 1, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 1, 1, 0, 0],
       [0, 1, 0, 1, 0, 2, 1, 0, 0, 0]])

In [94]:
cv_unigram_bigram.vocabulary_

{'what': 8,
 'is': 1,
 'nlp': 5,
 'what is': 9,
 'is nlp': 3,
 'great': 0,
 'nlp is': 6,
 'is great': 2,
 'popular': 7,
 'is popular': 4}

* **ngram_range**
- (2, 2) : only bigram
- (1, 2) : unigrams and bigrams both
- (1, 3) : unigrams, bigrams and trigrams

In [95]:
cv_bigram.fit_transform(["You are good", "You are not good"]).toarray()

array([[1, 0, 0, 1],
       [0, 1, 1, 1]])

### Advantages
- now You are good. and You are not good. both vectors are far away
- intuitive explanation and easy to implement

### Disadvantages
- if dataset is very large then it becomes slow
- OOV is still there , it won't throw error but it ignores new word


# 4. Tf-Idf

- instead of giving same weights to all the words which are present in a document it gives different weights based of TF and IDF calculation
- TF - Term frequency , how is the occurrence of given term in a document 
- TF(t, d) - (no of occurrences of term t in document d)/(total number of terms in document d)
- IDF - inverse document frequency, how is the occurrence of given term(word) in corpus
- IDF(t, d) = loge((total documents in corpus)/(no. of documents with term t in them))
- weight(t, d) = TF(t, d) * IDF(t, d)

In [96]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [97]:
tfidf.fit_transform(df['text']).toarray() # it returns sparse matrix, we can convert it to array using toarray() method

array([[0.        , 0.41988018, 0.41988018, 0.        , 0.8046125 ],
       [0.8046125 , 0.41988018, 0.41988018, 0.        , 0.        ],
       [0.        , 0.41988018, 0.41988018, 0.8046125 , 0.        ],
       [0.        , 0.4472136 , 0.89442719, 0.        , 0.        ]])

In [98]:
tfidf.vocabulary_

{'what': 4, 'is': 1, 'nlp': 2, 'great': 0, 'popular': 3}

### Advantages
- mostly used in information retrieval systems
- in google
### Disadvantes
- sparsity
- OOV
- dimensionality more --> overfitting
- no semantic meaning capturing

# 5. Custom features

- based on domain , sometimes we create our own features
- example : in sentiment analysis of a product based on reviews
- here we can make features like 1. no of positive words 2. no of negative words etc

------------------------------------------------------------------------------------------------------------------------------------------------

- Here i used stopwords, but actually we dont' consider stop words before vectorization.
- See my text preprocessing.ipynb notebook of Day - 1
- u can see this blog as well for text preprocessing and can see github
- https://vinod-codes-ai.blogspot.com/2026/05/day-2-nlp-text-preprocessing-python.html
- https://github.com/vinod-kaumar/NLP-by-vinod

# Assignment
- use movies.csv file which u have created in Data_Acquistion.ipynb of Day-1
- do all the step which are explained above|